# Hasnain, Han & Economo (2025) on the MAP dataset, DANDI:000363

Paper: Hasnain M. A., Han Y. & Economo M. N., "Mapping neural subspace dynamics onto the structure
of the mouse descending motor system", bioRxiv doi:10.1101/2025.10.17.682917 (preprint version 1).
The claims file `claims/hasnain-2025.yaml` was written from the preprint's full text (PMC12632980);
it fixes what is tested.

Dataset: DANDI:000363 (Mesoscale Activity Map Dataset, Chen et al. 2024), version **0.231012.2129**,
streamed from the archive. The paper cites the dandiset without a version
("https://dandiarchive.org/dandiset/000363"), so the most recent published version is pinned. The
earlier versions hold the same 174 sessions; the versions before 0.231012.2129 total 19 to 67 GB and
this one 65.7 TB, as it adds the raw recordings. The cell "Version check" compares the units of one
session between this version and 0.230822.0128.

The paper reuses the MAP data for Figure 4 only. Its own recordings (Figures 1 to 3, Extended Data
Figures 2 and 3) are not public, so those eleven claims are blocked.

| Claim | Tested here |
|---|---|
| own-recordings-cohort, fig1i-selectivity-proportions, fig2c-subspace-variance, fig2e-potent-only-units, fig2f-letn-preference-like-un, fig2gh-letn-span-subspaces, ed2f-false-negatives, fig3c-network-beats-linear, fig3d-letn-un-movement-encoding, ed3-letn-fast-switch, ed3a-cd-early-un-not-letn | Blocked: the paper's own recordings are not public |
| map-session-selection | Tested |
| fig4b-medulla-selectivity | Tested |
| fig4c-alm-null-potent-similar | Tested |
| fig4c-medulla-null-smaller | Tested |
| fig4d-alm-predicts-medulla-null | Tested |
| fig4d-decoding-asymmetry | Tested |
| fig4e-medulla-null-lower-dimension | Tested |
| fig4f-medulla-null-not-spanning-alm | Tested |

The methods are those of the claims file (header items M1 to M10, which cite the paper) and, where
it lacks a detail, the paper's Methods; the helper `map_subspace.py` says where each parameter comes
from.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import collections
import resource
import time
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import map_subspace as ms

T0 = time.time()
KEY = "hasnain-2025"


def seed(claim, k=0):
    return zlib.crc32(f"{KEY}/{claim}".encode()) + k


SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Data check

Every file's units table is read (probe target from the electrodes' `location`, `classification`,
`unit_quality`, `anno_name`), then the sessions are selected as in M1 and M2.

In [ ]:
assets = ms.asset_urls()
tables = ms.survey(assets)
print("files:", len(tables), " subjects:", len({t["subject"] for t in tables.values()}))
n_units = sum(len(t["target"]) for t in tables.values())
print("units:", n_units)
cls_q = collections.Counter((c, q) for t in tables.values() for c, q in zip(t["classification"], t["quality"]))
print("classification x unit_quality:", dict(cls_q))
anno_by_cls = collections.Counter((c, a != "") for t in tables.values() for c, a in zip(t["classification"], t["anno"]))
print("(classification, has CCF annotation):", dict(anno_by_cls))
targets = collections.Counter(tg for t in tables.values() for tg in set(t["target"]))
print("sessions per probe target:", dict(targets))
both_alm = [p for p, t in tables.items() if {"left ALM", "right ALM"} <= set(t["target"])]
alm_medulla_probes = [p for p, t in tables.items()
                      if any("ALM" in x for x in t["target"]) and any("Medulla" in x for x in t["target"])]
print("sessions with probes in both ALMs:", len(both_alm), "from", len({p.split('/')[0] for p in both_alm}), "mice")
print("sessions with an ALM and a medulla probe:", len(alm_medulla_probes), "from",
      len({p.split('/')[0] for p in alm_medulla_probes}), "mice")
anno_counts = collections.Counter(a for t in tables.values() for a in t["anno"] if a)
print("units in the medullary nuclei of M2:", {k: anno_counts[k] for k in ms.MEDULLA_NUCLEI})
print("units labelled MOs (any probe):", sum(v for k, v in anno_counts.items() if k.startswith("Secondary motor area")))
mos_off_alm = sum(1 for t in tables.values() for tg, a in zip(t["target"], t["anno"])
                  if a.startswith("Secondary motor area") and "ALM" not in tg)
print("units labelled MOs on probes not targeting ALM:", mos_off_alm)

### Version check

One session's units in the pinned version and in 0.230822.0128, the last version without the raw
recordings.

In [ ]:
from dandi.dandiapi import DandiAPIClient
probe_path = "sub-456774/sub-456774_ses-20191022T133136_behavior+ecephys+image+ogen.nwb"
old = DandiAPIClient().get_dandiset(ms.DANDISET_ID, "0.230822.0128")
old_paths = {a.path: a for a in old.get_assets()}
ses_key = lambda p: p.split("_behavior")[0]
old_by_ses = {ses_key(p): a for p, a in old_paths.items()}
print("assets in 0.230822.0128:", len(old_paths), " same paths as pinned:", len(set(old_paths) & set(tables)),
      " same sessions:", len(set(old_by_ses) & {ses_key(p) for p in tables}))
print("example path there:", next(p for p in old_paths if ses_key(p) == ses_key(probe_path)))
cmp = {}
for label, url in [("0.231012.2129", dict(assets)[probe_path]),
                   ("0.230822.0128", old_by_ses[ses_key(probe_path)].get_content_url(follow_redirects=1, strip_query=True))]:
    f = ms.open_file(url)
    u = f["units"]
    cmp[label] = dict(units=u["id"].shape[0], spikes=u["spike_times"].shape[0],
                      good=int(sum(np.array(ms._dec(u["classification"][:])) == "good")),
                      first_spikes=np.round(u["spike_times"][:5], 6).tolist(),
                      acquisition=sorted(f["acquisition"].keys()))
print(pd.DataFrame(cmp).T)

### Selection (M1, M2)

Units classified 'good' (`classification`), with a CCF label of MOs on a probe described as
targeting ALM, or a label of IRN, GRN or PARN. `anno_name` is filled only for units classified good,
so no other reading of "classified as 'good'" can place units in the CCF.

In [ ]:
bilateral, alm_med = ms.select_sessions(tables)
rows = []
for p, s in bilateral.items():
    rows.append(dict(kind="bilateral ALM", session=p.split("/")[1][:30], subject=p.split("/")[0],
                     right_ALM=len(s["right ALM"]), left_ALM=len(s["left ALM"])))
for p, s in alm_med.items():
    rows.append(dict(kind="ALM and medulla", session=p.split("/")[1][:30], subject=p.split("/")[0],
                     ALM=len(s["ALM"]), ALM_side="+".join(s["alm_sides"]), medulla=len(s["medulla"])))
sel_table = pd.DataFrame(rows)
print(sel_table.to_string())

### Trials, movement and clocks

For each selected session: the trials kept (hit or miss, no photostimulation, no early lick), the
go cue inside each trial, the jaw and nose speed and the movement threshold. Clock checks: the
tongue becomes visible to the side camera, and the first lick is logged, shortly after the go cue
(behaviour clock against trial table); and lick direction is decoded from the selected units' spike
counts in 200 ms windows around the go cue, against the same decoder with the trial labels shuffled
(spike clock against trial table).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

urls = dict(assets)
W = ms.WINDOWS["W1"]
sessions = {}
check_rows = []
rng_check = np.random.default_rng(seed("map-session-selection"))
for p in list(alm_med) + list(bilateral):
    if p in sessions:
        continue
    regions = alm_med[p] if p in alm_med else bilateral[p]
    ids = sorted(set(sum((v for k, v in regions.items() if k != "alm_sides"), [])))
    s = ms.load_session(urls[p], ids)
    gos = ms.go_times(s)
    keep = ms.trial_mask(s)
    go = gos[keep]
    speed = ms.speed_bins(s, go, W)
    finite = speed[np.isfinite(speed) & (speed > 0)]
    log_thr = ms.otsu_threshold(np.log10(finite))
    # tongue visibility and first lick
    f = ms.open_file(urls[p])
    tg = f["acquisition/BehavioralTimeSeries/Camera0_side_TongueTracking"]
    tts, tlik = tg["timestamps"][:], tg["data"][:, 2]
    licks = np.sort(np.r_[f["acquisition/BehavioralEvents/left_lick_times/timestamps"][:],
                          f["acquisition/BehavioralEvents/right_lick_times/timestamps"][:]])
    hit = np.array(s["trials"]["outcome"])[keep] == "hit"
    tongue_lat, lick_lat, tongue_pre = [], [], []
    for g in go[hit]:
        lo, hi = np.searchsorted(tts, [g, g + 1.0])
        k = np.where(tlik[lo:hi] > 0.9)[0]
        tongue_lat.append(tts[lo + k[0]] - g if len(k) else np.nan)
        lo2, hi2 = np.searchsorted(tts, [g - 0.5, g])
        tongue_pre.append(np.mean(tlik[lo2:hi2] > 0.9) if hi2 > lo2 else np.nan)
        j = np.searchsorted(licks, g)
        lick_lat.append(licks[j] - g if j < len(licks) else np.nan)
    # decoding lick direction from spike counts
    right = np.array(s["trials"]["trial_instruction"])[keep] == "right"
    acc = {}
    for w0 in (-2.0, -1.0, -0.4, 0.2, 0.8):
        X = np.stack([np.diff(np.searchsorted(s["spikes"][i], np.c_[go + w0, go + w0 + 0.2]), axis=1)[:, 0]
                      for i in ids], axis=1)
        clf = LogisticRegression(C=0.1, max_iter=2000)
        acc[w0] = cross_val_score(clf, X, right, cv=5).mean()
        acc[f"{w0} shuffled"] = cross_val_score(clf, X, rng_check.permutation(right), cv=5).mean()
    sessions[p] = dict(sess=s, keep=keep, go=go, speed=speed, log_thr=log_thr, regions=regions)
    check_rows.append(dict(session=p.split("/")[1][:30], trials=len(keep), kept=int(keep.sum()),
                           hits=int(hit.sum()), right=int(right.sum()),
                           go_in_trial=int(np.isfinite(gos).sum()),
                           speed_thr=round(10 ** log_thr, 1),
                           tongue_lat_ms=round(1000 * np.nanmedian(tongue_lat)),
                           tongue_pre_go=round(np.nanmean(tongue_pre), 3),
                           lick_lat_ms=round(1000 * np.nanmedian(lick_lat)),
                           **{f"acc {k}": round(v, 2) for k, v in acc.items()}))
    del f
checks = pd.DataFrame(check_rows)
print(checks.to_string())
print("trials kept per session: %d to %d" % (checks.kept.min(), checks.kept.max()))
print("median tongue latency over sessions: %d ms; first lick %d ms" % (checks.tongue_lat_ms.median(), checks.lick_lat_ms.median()))
print("elapsed %.0f s" % (time.time() - T0))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
tax = W[0] + ms.BIN * np.arange(sessions[probe_path]["speed"].shape[1])
spd_all = np.array([np.nanmean(v["speed"], 0) for v in sessions.values()])
ax[0].plot(tax, spd_all.T, lw=0.6)
ax[0].axvline(0, color="k", lw=0.5)
ax[0].set(xlabel="time from go cue (s)", ylabel="jaw and nose speed (pixels/s)", title="mean per session")
v = sessions[probe_path]
ax[1].hist(np.log10(v["speed"][np.isfinite(v["speed"]) & (v["speed"] > 0)]), 100)
ax[1].axvline(v["log_thr"], color="r")
ax[1].set(xlabel="log10 speed", title="one session, Otsu threshold in red")
plt.tight_layout()
print("mean speed range over sessions and bins: %.1f to %.1f pixels/s" % (np.nanmin(spd_all), np.nanmax(spd_all)))
print("speed in", probe_path.split('/')[1][:30], "pre-go %.1f, 0 to 1 s after go %.1f pixels/s" % (
    np.nanmean(v["speed"][:, (tax > -1) & (tax < 0)]), np.nanmean(v["speed"][:, (tax > 0) & (tax < 1)])))

## Claim map-session-selection

> In the MAP dataset, the paper's selection gives 20 bilateral ALM sessions from 10 mice (961 right
> and 1055 left ALM units) and ALM and medulla sessions with 604 ALM units and 724 medullary
> reticular nucleus units.

Criterion: session counts match S1 (16) or S2 (11) for ALM and medulla and 20 bilateral, and every
unit count within tolerance under one reading of M2. U1 (good units in the regions) is the main
reading. U2 (all units in those regions) cannot place units in the CCF, since only good units carry
a CCF label; its nearest form, all units on the region's probe, is shown.

In [ ]:
bil = sel_table[sel_table.kind == "bilateral ALM"]
am = sel_table[sel_table.kind == "ALM and medulla"]
measured_sel = dict(bilateral_sessions=len(bil), bilateral_mice=bil.subject.nunique(),
                    right_ALM=int(bil.right_ALM.sum()), left_ALM=int(bil.left_ALM.sum()),
                    alm_med_sessions=len(am), alm_med_mice=am.subject.nunique(),
                    ALM=int(am.ALM.sum()), medulla=int(am.medulla.sum()))
expected_sel = dict(bilateral_sessions=(20, 2), right_ALM=(961, 96.1), left_ALM=(1055, 105.5),
                    ALM=(604, 60.4), medulla=(724, 72.4))
for k, (v, tol) in expected_sel.items():
    print(f"{k:20s} measured {measured_sel[k]:6d}  expected {v} +- {tol}  within: {abs(measured_sel[k] - v) <= tol}")
print("bilateral mice", measured_sel["bilateral_mice"], "(paper 10); ALM and medulla sessions",
      measured_sel["alm_med_sessions"], "from", measured_sel["alm_med_mice"], "mice (paper S1 16, S2 11 from 7)")
print("bilateral units", measured_sel["right_ALM"] + measured_sel["left_ALM"], "(Methods 2016, Results 1116)")

# U2-like variant: every unit on the probe, sessions as selected
var_rows = []
for p, s in list(bilateral.items()) + list(alm_med.items()):
    t = tables[p]
    tg = np.array(t["target"])
    q = np.array(t["quality"])
    row = dict(session=p.split("/")[1][:30], kind="bilateral" if p in bilateral else "ALM and medulla")
    for side in ("left ALM", "right ALM", "left Medulla", "right Medulla"):
        row[side + " all"] = int((tg == side).sum())
        row[side + " quality good"] = int(((tg == side) & (q == "good")).sum())
    var_rows.append(row)
var = pd.DataFrame(var_rows).groupby("kind").sum(numeric_only=True)
print(var.T)

# thresholds on the count of good units: sessions selected
for thr in (20, 25, 30):
    b2, a2 = ms.select_sessions(tables, min_units=thr)
    print(f"at least {thr} good units: bilateral {len(b2)} sessions, ALM and medulla {len(a2)} sessions")

**Result: not reproduced.** With the dataset's CCF labels, the paper's rule gives 17 bilateral ALM
sessions from 9 mice (paper 20 from 10) and 5 ALM and medulla sessions from 4 mice (paper 16 or 11
from 7). The left ALM count (1001) is within tolerance of 1055; the right ALM count (858 against 961),
the ALM count of the ALM and medulla sessions (300 against 604) and the medulla count (340 against
724) are not. Only units classified good carry a CCF label, so the reading with all units of a region
cannot be formed; the nearest form, every unit on the region's probe, gives counts several times the
paper's (3668 units on the medulla probes of the 5 sessions). With a threshold of 20 good units the
ALM and medulla sessions number 14, with 25 they number 9.

## Per-region subspaces, ALM and medulla sessions

For each ALM and medulla session and region: z-scored rates, the movement mask, the dimensions
from the shuffle, and the subspaces. The main reading is W1, B1, D1 and the baseline z-score;
the readings W2, B2, D2 and the z-score of example.m are each run with the others held at the main
reading. The subspace generator uses the seed of fig4c-alm-null-potent-similar, the first claim
that uses it.

In [ ]:
READINGS = {"main": dict(W="W1", B="B1", D="D1", Z="baseline"),
            "W2": dict(W="W2", B="B1", D="D1", Z="baseline"),
            "B2": dict(W="W1", B="B2", D="D1", Z="baseline"),
            "D2": dict(W="W1", B="B1", D="D2", Z="baseline"),
            "Zall": dict(W="W1", B="B1", D="D1", Z="all")}


def prepare(p, region_ids, reading, keep_mask=None):
    v = sessions[p]
    s, go = v["sess"], v["go"]
    win = ms.WINDOWS[reading["W"]]
    r = ms.rates([s["spikes"][i] for i in region_ids], go, ms.WINDOWS["W1"])
    off = int(round((win[0] - ms.WINDOWS["W1"][0]) / ms.BIN))
    r = r[:, off:]
    if reading["Z"] == "baseline":
        # the baseline lies before W2's start, so it is always taken from the W1 rates
        rw1 = ms.rates([s["spikes"][i] for i in region_ids], go, ms.WINDOWS["W1"]) if off else r
        b0 = int(round((ms.BASELINE[0] - ms.WINDOWS["W1"][0]) / ms.BIN))
        b1 = int(round((ms.BASELINE[1] - ms.WINDOWS["W1"][0]) / ms.BIN))
        base = rw1[:, b0:b1].reshape(-1, r.shape[2])
        mu, sd = base.mean(0), base.std(0)
        allsd = r.reshape(-1, r.shape[2]).std(0)
        nfb = int((sd <= 1e-9).sum())
        sd = np.where(sd <= 1e-9, np.where(allsd > 0, allsd, 1.0), sd)
        z = ((r - mu) / sd).astype(np.float32)
    else:
        z, nfb = ms.zscore(r, win, mode="all")
    mv = ms.movement_mask(v["speed"][:, off:], v["log_thr"], short_runs_move=reading["B"] == "B2")
    return z, mv, nfb


fits = {}        # (session, region, reading) -> fit
fit_rows = []
rng_sub = np.random.default_rng(seed("fig4c-alm-null-potent-similar"))
for p, s in alm_med.items():
    for reg in ("ALM", "medulla"):
        dims_main = None
        for rname, rd in READINGS.items():
            z, mv, nfb = prepare(p, s[reg], rd)
            dims = fits[(p, reg, "main")]["d_raw"] if rname == "D2" else None
            fit = ms.fit_region(z, mv, rng_sub, dim_mode=rd["D"], dims=dims)
            fit["n_fallback"] = nfb
            fit["moving_frac"] = float((mv == 1).sum() / (mv >= 0).sum())
            if rname == "main":
                fit["z"] = z
                fit["mv"] = mv
            fits[(p, reg, rname)] = fit
            ve = ms.ve_readings(fit)
            fit_rows.append(dict(session=p.split("/")[1][:30], region=reg, reading=rname, units=z.shape[2],
                                 d_null=fit["d"][0], d_pot=fit["d"][1], d_null_raw=fit["d_raw"][0],
                                 d_pot_raw=fit["d_raw"][1], moving=round(fit["moving_frac"], 3),
                                 fallback=nfb, z_absmax=float(np.abs(z).max()),
                                 **{f"{k} {n}": 100 * x for k, vv in ve.items() for n, x in zip(("null", "pot", "both"), vv)}))
    print(p.split("/")[1][:30], "done, elapsed %.0f s" % (time.time() - T0))
fit_df = pd.DataFrame(fit_rows)
print(fit_df.round(1).to_string())
print("peak memory %.0f MB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3))

## Claim fig4b-medulla-selectivity

> Medullary reticular nucleus units are selective for lick direction in each epoch, most in the
> response epoch (sample 7%, delay 21%, response 49% of 724 units).

Correct trials only, equal random samples of right and left trials (the smaller count; M7), AUROC in
each of the five epochs, 1000 label shuffles. Readings E1 (a unit counts for an epoch if selective
in either sub-epoch) and E2 (each sub-epoch alone); Bonferroni (p < 0.01) and uncorrected (p < 0.05).
Firing rates are the smoothed 5 ms rates (not z-scored) averaged over the epoch's bins.

In [ ]:
def selectivity(rng):
    out = []
    for p, s in alm_med.items():
        v = sessions[p]
        trials = v["sess"]["trials"]
        out_arr = np.array(trials["outcome"])[v["keep"]]
        instr = np.array(trials["trial_instruction"])[v["keep"]]
        r_idx = np.where((out_arr == "hit") & (instr == "right"))[0]
        l_idx = np.where((out_arr == "hit") & (instr == "left"))[0]
        n = min(len(r_idx), len(l_idx))
        pick = np.r_[rng.choice(r_idx, n, replace=False), rng.choice(l_idx, n, replace=False)]
        is_right = np.r_[np.ones(n, bool), np.zeros(n, bool)]
        r = ms.rates([v["sess"]["spikes"][i] for i in s["medulla"]], v["go"][pick], ms.WINDOWS["W1"])
        tax = ms.WINDOWS["W1"][0] + ms.BIN * np.arange(r.shape[1])
        for j in range(r.shape[2]):
            row = dict(session=p.split("/")[1][:30], unit=s["medulla"][j], n_per_side=n)
            for ep, (a, b) in ms.EPOCHS.items():
                x = r[:, (tax >= a) & (tax < b), j].mean(1)
                auc, pv = ms.selectivity_p(x, is_right, rng)
                row[ep] = pv
                row[ep + "_auc"] = auc
            out.append(row)
    return pd.DataFrame(out)


def summarize_sel(df, alpha):
    sig = {ep: df[ep] < alpha for ep in ms.EPOCHS}
    e1 = {"sample": sig["sample"], "delay": sig["early_delay"] | sig["late_delay"],
          "response": sig["early_response"] | sig["late_response"]}
    res = {f"E1 {k}": 100 * v.mean() for k, v in e1.items()}
    res.update({f"E2 {k}": 100 * v.mean() for k, v in sig.items()})
    return res


sel_runs = {}
for k in SEED_OFFSETS:
    sel_runs[k] = selectivity(np.random.default_rng(seed("fig4b-medulla-selectivity", k)))
sel_main = sel_runs[0]
print("medulla units:", len(sel_main), " trials per side per session:", sel_main.groupby("session").n_per_side.first().to_dict())
tab = pd.DataFrame({(off, a): summarize_sel(df, a) for off, df in sel_runs.items() for a in (0.01, 0.05)}).T
tab.index.names = ["seed offset", "alpha"]
print(tab.round(1).to_string())
print("expected: sample 7 +- 0.7, delay 21 +- 2.1, response 49 +- 4.9")
by_sess = sel_main.groupby("session").apply(lambda d: pd.Series(summarize_sel(d, 0.01)))
print("by session (main seed, Bonferroni):")
print(by_sess.round(1).to_string())

**Result: partially reproduced.** Of 340 medulla units, under E1 with the Bonferroni threshold 34.1%
are selective in the sample epoch, 52.9% in the delay epoch and 86.5% in the response epoch (paper 7,
21 and 49%). The order holds (response highest, sample lowest) under every reading and seed, but no
reading brings a value within tolerance: the E2 sub-epochs give 31.2 to 67.4%, and the uncorrected
threshold gives higher values. Over the five seeds the E1 Bonferroni values range from 31.8 to 34.7%
(sample), 51.5 to 52.9% (delay) and 85.9 to 87.1% (response). The selection holds 340 units where the
paper has 724, so the units differ from the paper's.

## Claims fig4c-alm-null-potent-similar and fig4c-medulla-null-smaller

Variance explained per session (M6) under V1, V2 and V3, for each reading of M3 to M5; paired
two-sided t test of null against potent over the ALM and medulla sessions.

In [ ]:
ve_rows = []
for (rname, reg), d in fit_df.groupby(["reading", "region"]):
    for V in ("V1", "V2", "V3"):
        a, b = d[f"{V} null"].values, d[f"{V} pot"].values
        t, pv = stats.ttest_rel(a, b)
        ve_rows.append(dict(region=reg, reading=rname, V=V, n=len(a), null_mean=a.mean(), null_sd=a.std(ddof=1),
                            pot_mean=b.mean(), pot_sd=b.std(ddof=1), diff=(a - b).mean(), t=t, p=pv,
                            both=d[f"{V} both"].mean()))
ve_tab = pd.DataFrame(ve_rows).sort_values(["region", "reading", "V"])
print(ve_tab.round(3).to_string())
print("expected ALM null 36.0 +- 7.4, potent 37.1 +- 8.7 (p = 0.15); medulla null 26.4 +- 3.9, potent 44.8 +- 7.12 (p = 1.9e-9)")
alm_ok = ve_tab[(ve_tab.region == "ALM") & (abs(ve_tab.null_mean - 36.0) <= 7.4) & (abs(ve_tab.pot_mean - 37.1) <= 8.7)]
med_ok = ve_tab[(ve_tab.region == "medulla") & (abs(ve_tab.null_mean - 26.4) <= 3.9) & (abs(ve_tab.pot_mean - 44.8) <= 7.12)]
print("ALM readings with both means in tolerance:")
print(alm_ok[["reading", "V", "null_mean", "pot_mean", "p"]].round(4).to_string())
print("medulla readings with both means in tolerance:")
print(med_ok[["reading", "V", "null_mean", "pot_mean", "p"]].round(4).to_string())
print("per session, main reading:")
print(fit_df[fit_df.reading == "main"][["session", "region", "units", "d_null", "d_pot", "moving",
                                         "V1 null", "V1 pot", "V2 null", "V2 pot", "V3 null", "V3 pot"]].round(1).to_string())

Seed check for fig4c and fig4e: the main reading and reading B2 again under the claim's seed plus
1000 to 4000 (dimension shuffles and the optimizer's start).

In [ ]:
seed_rows = []
for rname in ("main", "B2"):
    for k in SEED_OFFSETS:
        rng_k = np.random.default_rng(seed("fig4c-alm-null-potent-similar", k))
        for p, s in alm_med.items():
            for reg in ("ALM", "medulla"):
                f0 = fits[(p, reg, "main")]
                if k == 0:
                    fit = fits[(p, reg, rname)]
                else:
                    mv = f0["mv"] if rname == "main" else ms.movement_mask(sessions[p]["speed"], sessions[p]["log_thr"], short_runs_move=True)
                    fit = ms.fit_region(f0["z"], mv, rng_k)
                ve = ms.ve_readings(fit)
                seed_rows.append(dict(reading=rname, offset=k, session=p, region=reg, d_null=fit["d"][0], d_pot=fit["d"][1],
                                      d_null_raw=fit["d_raw"][0],
                                      **{f"{V} {n}": 100 * x for V, vv in ve.items() for n, x in zip(("null", "pot", "both"), vv)}))
seed_df = pd.DataFrame(seed_rows)
srows = []
for (rname, off, reg), d in seed_df.groupby(["reading", "offset", "region"]):
    for V in ("V1", "V2", "V3"):
        t, pv = stats.ttest_rel(d[f"{V} null"], d[f"{V} pot"])
        srows.append(dict(reading=rname, offset=off, region=reg, V=V, null=d[f"{V} null"].mean(), pot=d[f"{V} pot"].mean(), p=pv,
                          mean_d_null=d.d_null.mean()))
seed_tab = pd.DataFrame(srows)
print(seed_tab.round(4).to_string())
print("elapsed %.0f s" % (time.time() - T0))

**Result (fig4c-alm-null-potent-similar): partially reproduced.** Over the 5 ALM and medulla
sessions, the paired t test finds no difference between null and potent variance in ALM under the
main reading (V1 54.5 against 77.3%, p = 0.126; V2 24.0 against 34.1%, p = 0.181; V3 p = 0.245), nor
under W2 or D2, but no reading of M3 to M6 puts both means within tolerance of 36.0 and 37.1%. Under
B2 the test finds potent above null (V1 p = 0.024, V2 p = 0.023), with means out of tolerance. Only
the z-score of example.m, a variant in the claims file, matches in full (V2 30.7 against 29.1%,
p = 0.119). With 5 sessions the test has little power: the main-reading V1 difference is 22.8 points.
Over the five seeds the main-reading V1 p ranges from 0.099 to 0.126, and the B2 V1 p from 0.023 to
0.027.

**Result (fig4c-medulla-null-smaller): reproduced, resting on reading B2.** Under B2 (supra-threshold
runs shorter than 100 ms counted as moving) with V2, the medulla null subspace explains 28.5% and the
potent subspace 51.2% (tolerances 26.4 +- 3.9 and 44.8 +- 7.12), paired t p = 0.0097, the same under the
five seeds. Under the main reading B1 the difference is not significant (V1 54.6 against 66.9%,
p = 0.445; V2 p = 0.380), nor under W2 or D2; under the example.m z-score V2 gives p = 0.042 with the
null mean outside tolerance. The criterion accepts any one reading, so the claim is reproduced under
B2, which also makes the ALM difference significant.

## Claim fig4e-medulla-null-lower-dimension

> The movement-null subspace has lower dimensionality in the medullary reticular nuclei than in ALM
> (paired t test, p = 7.4e-5).

N1: the shuffle dimension of the stationary activity (before any cap); N2: the participation ratio
of the covariance of the movement-null projections. Paired two-sided t test over the ALM and
medulla sessions.

In [ ]:
dim_rows = []
for p, s in alm_med.items():
    row = dict(session=p.split("/")[1][:30])
    for reg in ("ALM", "medulla"):
        fit = fits[(p, reg, "main")]
        row[f"{reg} units"] = len(s[reg])
        row[f"{reg} N1"] = fit["d_raw"][0]
        row[f"{reg} N2"] = ms.participation_ratio(fit["qn"].T @ fit["C"] @ fit["qn"])
        row[f"{reg} N2 per unit"] = row[f"{reg} N2"] / len(s[reg])
    dim_rows.append(row)
dim_df = pd.DataFrame(dim_rows)
print(dim_df.round(2).to_string())
for m in ("N1", "N2", "N2 per unit"):
    t, pv = stats.ttest_rel(dim_df[f"medulla {m}"], dim_df[f"ALM {m}"])
    print(f"{m}: medulla mean {dim_df[f'medulla {m}'].mean():.2f}, ALM mean {dim_df[f'ALM {m}'].mean():.2f}, "
          f"t = {t:.2f}, p = {pv:.3g}, medulla lower in {(dim_df[f'medulla {m}'] < dim_df[f'ALM {m}']).sum()} of {len(dim_df)}")
for off, d in seed_df[seed_df.reading == "main"].groupby("offset"):
    a = d[d.region == "ALM"].d_null_raw.values
    b = d[d.region == "medulla"].d_null_raw.values
    t, pv = stats.ttest_rel(b, a)
    print(f"seed offset {off}: N1 medulla {b.mean():.1f}, ALM {a.mean():.1f}, p = {pv:.3g}")
for rname in READINGS:
    d = fit_df[fit_df.reading == rname]
    a = d[d.region == "ALM"].d_null_raw.values
    b = d[d.region == "medulla"].d_null_raw.values
    t, pv = stats.ttest_rel(b, a)
    print(f"reading {rname}: N1 medulla {b.mean():.1f}, ALM {a.mean():.1f}, p = {pv:.3g}")

**Result: not reproduced.** The medulla's movement-null dimensionality is not lower than ALM's under
either reading. N1 (shuffle dimension): medulla 6.8, ALM 7.4, p = 0.646, lower in 2 of 5 sessions, and
p = 0.577 to 0.646 over the five seeds; the other readings of M3 to M5 give p = 0.077 to 0.646 (under
B2 the medulla is higher). N2 (participation ratio): medulla 3.13, ALM 5.09, p = 0.0915, lower in 4
of 5; divided by the unit count, p = 0.31. The medulla has fewer units than ALM in 2 sessions and more
in 3.

## Bilateral ALM sessions: subspaces for the decoding reference

Main reading only (W1, B1, D1, baseline z-score); each hemisphere's subspaces from its own units.

In [ ]:
for p, s in bilateral.items():
    for reg in ("right ALM", "left ALM"):
        z, mv, nfb = prepare(p, s[reg], READINGS["main"])
        fit = ms.fit_region(z, mv, rng_sub)
        fit["proj"] = (z @ fit["qn"]).astype(np.float32)
        fit["n_fallback"] = nfb
        fits[(p, reg, "main")] = fit
    print(p.split("/")[1][:30], [fits[(p, r, "main")]["d"] for r in ("right ALM", "left ALM")],
          "elapsed %.0f s" % (time.time() - T0))
for p, s in alm_med.items():
    for reg in ("ALM", "medulla"):
        f0 = fits[(p, reg, "main")]
        f0["proj"] = (f0["z"] @ f0["qn"]).astype(np.float32)
print("peak memory %.0f MB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3))

## Claims fig4d-alm-predicts-medulla-null and fig4d-decoding-asymmetry

Ridge (lambda 0.01) from the source's movement-null projections at t and the 8 preceding bins to
all target movement-null dimensions at t; 70/30 split of trials; R2 per bin from -1.8 to +1.5 s,
averaged (M8). Readings: T1 all trials, T2 correct only; R1 pooled R2, R2 mean over dimensions;
predictors standardized or not; A1 mean of both ALM directions, A2 each direction.

In [ ]:
bins = np.arange(int(round((ms.TASK_SPAN[0] - ms.WINDOWS["W1"][0]) / ms.BIN)),
                 int(round((ms.TASK_SPAN[1] - ms.WINDOWS["W1"][0]) / ms.BIN)))


def decode(rng, readings):
    rows = []
    for kind, sess_map, pairs in (("ALM-medulla", alm_med, [("ALM", "medulla"), ("medulla", "ALM")]),
                                  ("ALM-ALM", bilateral, [("right ALM", "left ALM"), ("left ALM", "right ALM")])):
        for p in sess_map:
            v = sessions[p]
            hit = np.array(v["sess"]["trials"]["outcome"])[v["keep"]] == "hit"
            for T in ("T1", "T2"):
                idx = np.arange(len(hit)) if T == "T1" else np.where(hit)[0]
                perm = rng.permutation(idx)
                ntr = int(round(0.7 * len(perm)))
                train, test = perm[:ntr], perm[ntr:]
                for src, tgt in pairs:
                    xs, xt = fits[(p, src, "main")]["proj"], fits[(p, tgt, "main")]["proj"]
                    for stdz, pooled in readings:
                        r2 = ms.ridge_r2(xs, xt, train, test, bins, stdz, pooled)
                        rows.append(dict(kind=kind, session=p, T=T, src=src, tgt=tgt,
                                         std=stdz, R="R1" if pooled else "R2", r2=100 * r2.mean(),
                                         d_src=xs.shape[2], d_tgt=xt.shape[2], n_train=len(train)))
    return pd.DataFrame(rows)


ALL_READ = [(s_, p_) for s_ in (False, True) for p_ in (True, False)]
dec_main = decode(np.random.default_rng(seed("fig4d-alm-predicts-medulla-null")), ALL_READ)


def dec_summary(df):
    out = []
    for (T, stdz, R), d in df.groupby(["T", "std", "R"]):
        am_ = d[(d.kind == "ALM-medulla") & (d.src == "ALM")].set_index("session").r2
        ma_ = d[(d.kind == "ALM-medulla") & (d.src == "medulla")].set_index("session").r2
        aa = d[d.kind == "ALM-ALM"]
        a1 = aa.groupby("session").r2.mean()
        rl = aa[aa.src == "right ALM"].r2
        lr = aa[aa.src == "left ALM"].r2
        sem = lambda x: x.std(ddof=1) / np.sqrt(len(x))
        t, pv = stats.ttest_rel(am_.loc[ma_.index], ma_)
        out.append(dict(T=T, std=stdz, R=R, ALM_to_med=am_.mean(), sem1=sem(am_), med_to_ALM=ma_.mean(),
                        sem2=sem(ma_), p_asym=pv, ALM_ALM_A1=a1.mean(), sem3=sem(a1), R_to_L=rl.mean(),
                        L_to_R=lr.mean()))
    return pd.DataFrame(out)


dsum = dec_summary(dec_main)
print(dsum.round(4).to_string())
print("expected ALM to medulla 40 +- 8.1, ALM to ALM 54 +- 7.0, medulla to ALM 21 +- 6.7 (p = 1.2e-4)")
print("per session, T1, unstandardized, R1:")
print(dec_main[(dec_main["T"] == "T1") & (~dec_main["std"]) & (dec_main.R == "R1")][
    ["kind", "session", "src", "tgt", "d_src", "d_tgt", "n_train", "r2"]].round(1).to_string())

In [ ]:
seed_dec = []
for k in SEED_OFFSETS:
    d = decode(np.random.default_rng(seed("fig4d-alm-predicts-medulla-null", k)), [(False, True), (True, True)]) if k else dec_main
    s_ = dec_summary(d)
    s_["offset"] = k
    seed_dec.append(s_[s_.R == "R1"])
seed_dec = pd.concat(seed_dec)
print(seed_dec[["offset", "T", "std", "ALM_to_med", "med_to_ALM", "p_asym", "ALM_ALM_A1"]].round(4).to_string())
print("elapsed %.0f s" % (time.time() - T0))

Variants, not readings of the claim: the claim fixes lambda = 0.01. The lagged projections of the
smoothed rates are nearly collinear, so the variants show the same decoding with larger penalties,
and the median over bins next to the mean (main seed plus 1, T1, standardized predictors, R1).

In [ ]:
p0 = next(iter(bilateral))
xl0 = ms.lagged(fits[(p0, "right ALM", "main")]["proj"])[:, bins[300]].astype(float)
print("condition number of the predictor correlation matrix, one bilateral session, one bin: %.3g"
      % np.linalg.cond(np.corrcoef(xl0.T)))
lam_rows = []
for lam in (0.01, 1.0, 100.0, 1e4):
    rng_l = np.random.default_rng(seed("fig4d-alm-predicts-medulla-null", 1))
    for kind, sess_map, pairs in (("ALM-medulla", alm_med, [("ALM", "medulla"), ("medulla", "ALM")]),
                                  ("ALM-ALM", bilateral, [("right ALM", "left ALM"), ("left ALM", "right ALM")])):
        for p in sess_map:
            perm = rng_l.permutation(len(sessions[p]["go"]))
            ntr = int(round(0.7 * len(perm)))
            for src, tgt in pairs:
                r2 = ms.ridge_r2(fits[(p, src, "main")]["proj"], fits[(p, tgt, "main")]["proj"],
                                 perm[:ntr], perm[ntr:], bins, True, True, lam=lam)
                lam_rows.append(dict(lam=lam, kind=kind, session=p, src=src, mean=100 * r2.mean(), median=100 * np.median(r2)))
lam_df = pd.DataFrame(lam_rows)
lam_df["dir"] = np.where(lam_df.kind == "ALM-ALM", "ALM to ALM", np.where(lam_df.src == "ALM", "ALM to medulla", "medulla to ALM"))
lam_sum = lam_df.groupby(["lam", "session", "dir"])[["mean", "median"]].mean().groupby(["lam", "dir"]).agg(["mean", "sem"])
print(lam_sum.round(2).to_string())
for lam, d in lam_df.groupby("lam"):
    a = d[d.dir == "ALM to medulla"].set_index("session")["mean"]
    b = d[d.dir == "medulla to ALM"].set_index("session")["mean"]
    t, pv = stats.ttest_rel(a, b.loc[a.index])
    print(f"lambda {lam:g}: ALM to medulla minus medulla to ALM {(a - b.loc[a.index]).mean():.2f} points, paired t p = {pv:.3g}")

**Result (fig4d-alm-predicts-medulla-null): not reproduced.** With lambda = 0.01 the decoder fails on
held-out trials under every reading: mean R2 from -228.5 to -149.8% for ALM to medulla and from
-7333 to -2712% for ALM to ALM (A1), against 40 and 54%, and it stays negative under the five seeds.
The nine lagged copies of the smoothed projections are nearly collinear (condition number about
1e6), and lambda = 0.01 barely regularizes them. In the variants with larger penalties R2 is at most
21.3% (ALM to medulla, lambda 100), and ALM to ALM (5.8%) stays below ALM to medulla, the reverse of
the paper.

**Result (fig4d-decoding-asymmetry): not reproduced.** Under every reading the paired t test of ALM to
medulla against medulla to ALM gives p = 0.46 to 0.90 over the 5 sessions, and medulla to ALM R2 is
-378 to -185%, against 21 +- 6.7%. In the lambda = 100 variant ALM to medulla exceeds medulla to ALM
by 13.5 points (p = 0.040), with medulla to ALM at 7.8%.

## Claim fig4f-medulla-null-not-spanning-alm

> In a joint UMAP embedding of movement-null activity, medullary units do not span the set of
> movement-null activity patterns seen in ALM.

M10: equal numbers of left and right correct trials, half to train and half to test; trial-averaged
movement-null reconstructions (X Q Q') of both trial types concatenated in time; z-scored per unit on
the training set; UMAP (2 components, 30 neighbours, min_dist 0.3, cosine), fitted on the training
data of ALM and medulla units together, test data transformed. The paper judges by inspection; the
numbers below describe the test embedding: for each unit, whether any of its 10 nearest neighbours
is from the other region, against the same count with the region labels shuffled (1000 times).

In [ ]:
import umap
from sklearn.neighbors import NearestNeighbors


def umap_run(rng, random_state):
    train_rows, test_rows, region = [], [], []
    for p, s in alm_med.items():
        v = sessions[p]
        trials = v["sess"]["trials"]
        out_arr = np.array(trials["outcome"])[v["keep"]]
        instr = np.array(trials["trial_instruction"])[v["keep"]]
        r_idx = rng.permutation(np.where((out_arr == "hit") & (instr == "right"))[0])
        l_idx = rng.permutation(np.where((out_arr == "hit") & (instr == "left"))[0])
        n = min(len(r_idx), len(l_idx))
        r_idx, l_idx = r_idx[:n], l_idx[:n]
        h = n // 2
        for reg in ("ALM", "medulla"):
            f0 = fits[(p, reg, "main")]
            rec = f0["z"] @ f0["qn"] @ f0["qn"].T
            tr = np.concatenate([rec[r_idx[:h]].mean(0), rec[l_idx[:h]].mean(0)], axis=0)
            te = np.concatenate([rec[r_idx[h:]].mean(0), rec[l_idx[h:]].mean(0)], axis=0)
            mu, sd = tr.mean(0), tr.std(0)
            sd = np.where(sd > 0, sd, 1.0)
            train_rows.append(((tr - mu) / sd).T)
            test_rows.append(((te - mu) / sd).T)
            region += [reg] * tr.shape[1]
    Xtr, Xte = np.vstack(train_rows), np.vstack(test_rows)
    model = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3, metric="cosine", random_state=random_state)
    model.fit(Xtr)
    return model.transform(Xte), np.array(region)


def nn_stats(emb, region, rng, k=10, n_shuffle=1000):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(emb)
    _, ind = nn.kneighbors(emb)
    ind = ind[:, 1:]

    def frac(lab):
        other = lab[ind] != lab[:, None]
        alm_alone = np.mean(~other[lab == "ALM"].any(1))
        med_with_alm = np.mean(other[lab == "medulla"].any(1))
        return alm_alone, med_with_alm

    obs = frac(region)
    null = np.array([frac(rng.permutation(region)) for _ in range(n_shuffle)])
    return obs, null


umap_res = {}
for k in SEED_OFFSETS:
    sd_ = seed("fig4f-medulla-null-not-spanning-alm", k)
    rng_u = np.random.default_rng(sd_)
    emb, region = umap_run(rng_u, random_state=sd_ % (2 ** 31))
    obs, null = nn_stats(emb, region, rng_u)
    umap_res[k] = (emb, region, obs, null)
    print(f"seed offset {k}: ALM units with no medulla unit among 10 neighbours {100 * obs[0]:.1f}% "
          f"(shuffled labels {100 * null[:, 0].mean():.1f}%, 95th pct {100 * np.percentile(null[:, 0], 95):.1f}%); "
          f"medulla units with an ALM neighbour {100 * obs[1]:.1f}% (shuffled {100 * null[:, 1].mean():.1f}%)")
emb, region, obs, null = umap_res[0]
fig, ax = plt.subplots(figsize=(5, 4.5))
for reg, c in (("ALM", "tab:blue"), ("medulla", "tab:orange")):
    ax.scatter(*emb[region == reg].T, s=6, c=c, label=f"{reg} ({(region == reg).sum()})", alpha=0.7)
ax.legend()
ax.set(title="Test-set UMAP of movement-null reconstructions", xlabel="UMAP 1", ylabel="UMAP 2")
print("embedding range: x %.2f to %.2f, y %.2f to %.2f; all points shown" % (emb[:, 0].min(), emb[:, 0].max(), emb[:, 1].min(), emb[:, 1].max()))
# extent: the share of the ALM points' 2-d spread covered by medulla points
for reg in ("ALM", "medulla"):
    e = emb[region == reg]
    print(reg, "5th to 95th percentile: x %.2f to %.2f, y %.2f to %.2f" % (*np.percentile(e[:, 0], [5, 95]), *np.percentile(e[:, 1], [5, 95])))
grid = np.floor((emb - emb.min(0)) / ((emb.max(0) - emb.min(0)) / 10 + 1e-12)).astype(int)
cells_alm = {tuple(g) for g, r in zip(grid, region) if r == "ALM"}
cells_med = {tuple(g) for g, r in zip(grid, region) if r == "medulla"}
print("10 x 10 grid cells holding ALM units: %d, of which hold no medulla unit: %d; medulla cells outside ALM cells: %d of %d"
      % (len(cells_alm), len(cells_alm - cells_med), len(cells_med - cells_alm), len(cells_med)))
print("elapsed %.0f s, peak memory %.0f MB" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3))

**Result: partially reproduced.** In the test-set embedding 20.0% of ALM units have no medulla unit
among their 10 nearest neighbours (shuffled labels 0.1%), 20.0 to 25.7% over the five seeds, so part
of the ALM movement-null patterns has no medulla counterpart, as the paper reports. The second part
of the criterion does not hold: only 55.9% of medulla units have an ALM unit among their neighbours
(shuffled 99.8%; 52.1 to 60.6% over the seeds), and 7 of the 39 grid cells that hold medulla units
hold no ALM unit, so the medulla units do not all fall within the region occupied by ALM units. The
judgment rests on these neighbour counts and the plot, as the paper's rests on inspection.

## Summary

| Claim | Outcome |
|---|---|
| Eleven claims on the paper's own recordings (Figures 1 to 3, Extended Data Figures 2 and 3) | Blocked: data missing |
| map-session-selection | Not reproduced |
| fig4b-medulla-selectivity | Partially reproduced |
| fig4c-alm-null-potent-similar | Partially reproduced |
| fig4c-medulla-null-smaller | Reproduced, resting on reading B2 |
| fig4d-alm-predicts-medulla-null | Not reproduced |
| fig4d-decoding-asymmetry | Not reproduced |
| fig4e-medulla-null-lower-dimension | Not reproduced |
| fig4f-medulla-null-not-spanning-alm | Partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's:

- The selection gives 5 ALM and medulla sessions (paper 16 or 11) and 17 bilateral sessions (paper
  20), so every Figure 4 test runs on fewer sessions and units than the paper's.
- The movement threshold is Otsu's threshold on the log speed, where the paper set it by hand; the
  keypoint positions are smoothed (3 frames) first.
- Ignore, photostimulation and early-lick trials are left out.
- The dimension shuffle uses 100 shuffles on every 4th bin (the paper 1000 on all bins).
- UMAP runs in umap-learn, not MATLAB's run_umap.
- The readings of the claims file were each run with the others held at the main reading, not in
  every combination.

What was found about the dataset:

- `anno_name` (the CCF label) is filled only for units whose `classification` is good (69453 of
  272227 units), and 1852 units have no `classification`. The paper's rule for the regions can only
  be applied to good units.
- Some units classified good have near-silent baselines, so the baseline z-score of the Methods gives
  values up to 213 s.d. in a region.
- The jaw and nose tracking are in pixels, at about 294 Hz, in one segment per trial; unsmoothed, their
  frame-to-frame jitter hides movement.
- Clocks: in every selected session the tongue becomes visible a median 56 to 271 ms after the go cue
  and is visible in at most 18% of the frames of the 0.5 s before it, and lick direction is decoded
  from spike counts after the go cue above the shuffled labels (0.63 to 0.97 at +0.8 s, shuffled 0.48
  to 0.61), so the spike, video and trial clocks are consistent.
- The units of the session compared are identical in versions 0.230822.0128 and 0.231012.2129.